# ECON 5193: Problem Set 4 — Instrumental Variables
## Empirical Analysis: Economic Growth and Civil Conflict in Sub-Saharan Africa
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white
from linearmodels.iv import IV2SLS
from scipy import stats

plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 11})
sns.set_style('whitegrid')

df = pd.read_csv('conflict_iv_data.csv')
print(f"Dataset: {df.shape[0]} observations, {df.shape[1]} variables")
print(f"Countries: {df['country_id'].nunique()}, Years: {df['year'].min()}-{df['year'].max()}")
df.head()

## Question 4: Data Exploration and OLS Baseline

### 4(a) Summary Statistics

In [ ]:
key_vars = ['conflict_onset', 'gdp_growth', 'rainfall_growth', 'rainfall_growth_lag',
            'ethnic_fractionalization', 'polity_score', 'oil_exporter']

summary = df[key_vars].describe().T[['mean', 'std', 'min', 'max']]
summary.columns = ['Mean', 'Std. Dev.', 'Min', 'Max']
summary = summary.round(3)

# Rename for readability
summary.index = ['Conflict Onset', 'GDP Growth (%)', 'Rainfall Growth (%)',
                  'Rainfall Growth Lag (%)', 'Ethnic Fractionalization',
                  'Polity Score', 'Oil Exporter']

print("Table 1: Summary Statistics")
print("=" * 65)
print(summary.to_string())
print(f"\nN = {len(df)}")
print(f"\nConflict prevalence: {df['conflict_onset'].mean()*100:.1f}% of country-years")
print(f"GDP growth range: {df['gdp_growth'].min():.1f}% to {df['gdp_growth'].max():.1f}%")
print(f"Mean GDP growth: {df['gdp_growth'].mean():.2f}%")

### 4(b) Naïve OLS Regression

In [ ]:
# Define controls
controls = ['log_population', 'ethnic_fractionalization', 'mountainous_terrain',
            'oil_exporter', 'polity_score']

# OLS with robust standard errors
X_ols = sm.add_constant(df[['gdp_growth'] + controls])
y = df['conflict_onset']

ols_model = sm.OLS(y, X_ols).fit(cov_type='HC1')

print("Table 2: OLS Regression — Conflict Onset on GDP Growth")
print("=" * 65)
print(ols_model.summary2().tables[1].to_string())
print(f"\nCoefficient on GDP Growth: {ols_model.params['gdp_growth']:.5f}")
print(f"Robust Std. Error: {ols_model.bse['gdp_growth']:.5f}")
print(f"t-statistic: {ols_model.tvalues['gdp_growth']:.3f}")
print(f"p-value: {ols_model.pvalues['gdp_growth']:.4f}")
print(f"R-squared: {ols_model.rsquared:.4f}")
print(f"N = {int(ols_model.nobs)}")

### 4(c) Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw scatter
axes[0].scatter(df['gdp_growth'], df['conflict_onset'], alpha=0.15, s=15, color='steelblue')
# Linear fit
z = np.polyfit(df['gdp_growth'], df['conflict_onset'], 1)
p = np.poly1d(z)
x_range = np.linspace(df['gdp_growth'].min(), df['gdp_growth'].max(), 100)
axes[0].plot(x_range, p(x_range), 'r-', linewidth=2, label=f'Linear fit (slope={z[0]:.4f})')
axes[0].set_xlabel('GDP Growth (%)')
axes[0].set_ylabel('Conflict Onset')
axes[0].set_title('(a) Raw Scatter: Conflict vs GDP Growth')
axes[0].legend()

# Binned scatter
df['gdp_bin'] = pd.cut(df['gdp_growth'], bins=20)
binned = df.groupby('gdp_bin', observed=True).agg(
    conflict_mean=('conflict_onset', 'mean'),
    gdp_mid=('gdp_growth', 'mean'),
    count=('conflict_onset', 'size')
).dropna()

axes[1].scatter(binned['gdp_mid'], binned['conflict_mean'], s=binned['count']*2,
                color='steelblue', edgecolors='navy', alpha=0.7, zorder=5)
z2 = np.polyfit(binned['gdp_mid'], binned['conflict_mean'], 1)
p2 = np.poly1d(z2)
x_range2 = np.linspace(binned['gdp_mid'].min(), binned['gdp_mid'].max(), 100)
axes[1].plot(x_range2, p2(x_range2), 'r-', linewidth=2, label=f'Linear fit (slope={z2[0]:.4f})')
axes[1].set_xlabel('GDP Growth (%) — Bin Midpoints')
axes[1].set_ylabel('Mean Conflict Rate')
axes[1].set_title('(b) Binned Scatter (20 bins)')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_q4c_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
df.drop(columns='gdp_bin', inplace=True)

## Question 5: The First Stage

### 5(a) First Stage — Single Instrument

In [ ]:
# First stage: gdp_growth on rainfall_growth + controls
X_fs1 = sm.add_constant(df[['rainfall_growth'] + controls])
y_fs = df['gdp_growth']

fs1_model = sm.OLS(y_fs, X_fs1).fit(cov_type='HC1')

print("Table 3: First-Stage Regression (Single Instrument)")
print("=" * 65)
print(fs1_model.summary2().tables[1].to_string())

# F-statistic for excluded instrument
# Use Wald test for the coefficient on rainfall_growth
f_test_1 = fs1_model.wald_test('rainfall_growth = 0', use_f=True)
f_stat_1 = f_test_1.statistic[0][0]
f_pval_1 = f_test_1.pvalue

print(f"\nCoefficient on Rainfall Growth: {fs1_model.params['rainfall_growth']:.5f}")
print(f"Robust Std. Error: {fs1_model.bse['rainfall_growth']:.5f}")
print(f"First-stage F-statistic (excluded instrument): {f_stat_1:.2f}")
print(f"F-test p-value: {f_pval_1:.6f}")
print(f"R-squared: {fs1_model.rsquared:.4f}")

### 5(b) First Stage — Two Instruments

In [ ]:
X_fs2 = sm.add_constant(df[['rainfall_growth', 'rainfall_growth_lag'] + controls])

fs2_model = sm.OLS(y_fs, X_fs2).fit(cov_type='HC1')

print("Table 4: First-Stage Regression (Two Instruments)")
print("=" * 65)
print(fs2_model.summary2().tables[1].to_string())

# Joint F-test for both excluded instruments
f_test_2 = fs2_model.wald_test('rainfall_growth = 0, rainfall_growth_lag = 0', use_f=True)
f_stat_2 = f_test_2.statistic[0][0]
f_pval_2 = f_test_2.pvalue

print(f"\nCoefficient on Rainfall Growth: {fs2_model.params['rainfall_growth']:.5f}")
print(f"Coefficient on Rainfall Growth (Lag): {fs2_model.params['rainfall_growth_lag']:.5f}")
print(f"Joint F-statistic (excluded instruments): {f_stat_2:.2f}")
print(f"F-test p-value: {f_pval_2:.6f}")
print(f"R-squared: {fs2_model.rsquared:.4f}")

### 5(c) First-Stage Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw scatter
axes[0].scatter(df['rainfall_growth'], df['gdp_growth'], alpha=0.2, s=15, color='forestgreen')
z = np.polyfit(df['rainfall_growth'], df['gdp_growth'], 1)
p = np.poly1d(z)
x_range = np.linspace(df['rainfall_growth'].min(), df['rainfall_growth'].max(), 100)
axes[0].plot(x_range, p(x_range), 'r-', linewidth=2, label=f'Linear fit (slope={z[0]:.4f})')
axes[0].set_xlabel('Rainfall Growth (%)')
axes[0].set_ylabel('GDP Growth (%)')
axes[0].set_title('(a) Raw Scatter: GDP Growth vs Rainfall Growth')
axes[0].legend()

# Binned scatter
df['rain_bin'] = pd.cut(df['rainfall_growth'], bins=20)
binned_fs = df.groupby('rain_bin', observed=True).agg(
    gdp_mean=('gdp_growth', 'mean'),
    rain_mid=('rainfall_growth', 'mean'),
    count=('gdp_growth', 'size')
).dropna()

axes[1].scatter(binned_fs['rain_mid'], binned_fs['gdp_mean'], s=binned_fs['count']*3,
                color='forestgreen', edgecolors='darkgreen', alpha=0.7, zorder=5)
z2 = np.polyfit(binned_fs['rain_mid'], binned_fs['gdp_mean'], 1)
p2 = np.poly1d(z2)
x_range2 = np.linspace(binned_fs['rain_mid'].min(), binned_fs['rain_mid'].max(), 100)
axes[1].plot(x_range2, p2(x_range2), 'r-', linewidth=2, label=f'Linear fit (slope={z2[0]:.4f})')
axes[1].set_xlabel('Rainfall Growth (%) — Bin Midpoints')
axes[1].set_ylabel('Mean GDP Growth (%)')
axes[1].set_title('(b) Binned Scatter (20 bins)')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_q5c_first_stage_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
df.drop(columns='rain_bin', inplace=True)

## Question 6: IV Estimation and Diagnostics

### 6(a) Reduced-Form Regression

In [ ]:
# Reduced form: conflict_onset on rainfall_growth + controls
X_rf = sm.add_constant(df[['rainfall_growth'] + controls])
y_rf = df['conflict_onset']

rf_model = sm.OLS(y_rf, X_rf).fit(cov_type='HC1')

print("Table 5: Reduced-Form Regression")
print("=" * 65)
print(rf_model.summary2().tables[1].to_string())
print(f"\nCoefficient on Rainfall Growth: {rf_model.params['rainfall_growth']:.6f}")
print(f"Robust Std. Error: {rf_model.bse['rainfall_growth']:.6f}")
print(f"p-value: {rf_model.pvalues['rainfall_growth']:.4f}")

# Wald ratio
wald_ratio = rf_model.params['rainfall_growth'] / fs1_model.params['rainfall_growth']
print(f"\nWald Ratio (reduced-form / first-stage): {wald_ratio:.5f}")

### 6(b) 2SLS IV — Single Instrument

In [ ]:
# Using linearmodels IV2SLS
# Formula: dependent ~ exogenous ~ endogenous
# IV2SLS(dependent, exog, endog, instruments)

exog_vars = sm.add_constant(df[controls])

iv1_model = IV2SLS(
    dependent=df['conflict_onset'],
    exog=exog_vars,
    endog=df[['gdp_growth']],
    instruments=df[['rainfall_growth']]
).fit(cov_type='robust')

print("Table 6: 2SLS IV Regression (Single Instrument: rainfall_growth)")
print("=" * 65)
print(iv1_model.summary)

print(f"\nIV Coefficient on GDP Growth: {iv1_model.params['gdp_growth']:.5f}")
print(f"Robust Std. Error: {iv1_model.std_errors['gdp_growth']:.5f}")
print(f"OLS Coefficient on GDP Growth: {ols_model.params['gdp_growth']:.5f}")
print(f"\nComparison: IV estimate is {'larger' if abs(iv1_model.params['gdp_growth']) > abs(ols_model.params['gdp_growth']) else 'smaller'} in absolute value than OLS")

### 6(c) 2SLS IV — Two Instruments + Overidentification Test

In [ ]:
iv2_model = IV2SLS(
    dependent=df['conflict_onset'],
    exog=exog_vars,
    endog=df[['gdp_growth']],
    instruments=df[['rainfall_growth', 'rainfall_growth_lag']]
).fit(cov_type='robust')

print("Table 7: 2SLS IV Regression (Two Instruments)")
print("=" * 65)
print(iv2_model.summary)

print(f"\nIV Coefficient on GDP Growth: {iv2_model.params['gdp_growth']:.5f}")
print(f"Robust Std. Error: {iv2_model.std_errors['gdp_growth']:.5f}")

In [ ]:
# Hansen J / Sargan overidentification test
# Manual computation: regress 2SLS residuals on all exogenous variables (controls + instruments)
resid_iv2 = iv2_model.resids

X_overid = sm.add_constant(df[controls + ['rainfall_growth', 'rainfall_growth_lag']])
overid_reg = sm.OLS(resid_iv2, X_overid).fit()

n = len(resid_iv2)
j_stat = n * overid_reg.rsquared  # Hansen J = n * R^2
j_df = 1  # number of overidentifying restrictions = num instruments - num endogenous = 2 - 1
j_pval = 1 - stats.chi2.cdf(j_stat, j_df)

print("Overidentification Test (Hansen J / Sargan)")
print("=" * 50)
print(f"J-statistic: {j_stat:.4f}")
print(f"Degrees of freedom: {j_df}")
print(f"p-value: {j_pval:.4f}")
print(f"\nNull hypothesis: All instruments are valid (satisfy exclusion restriction)")
if j_pval > 0.05:
    print(f"Result: Fail to reject at 5% level — no evidence against instrument validity")
else:
    print(f"Result: Reject at 5% level — evidence of instrument invalidity")

### 6(d) Hausman Test (Durbin-Wu-Hausman)

In [ ]:
# Durbin-Wu-Hausman test: augmented regression approach
# Step 1: Get first-stage residuals
X_fs_full = sm.add_constant(df[['rainfall_growth', 'rainfall_growth_lag'] + controls])
fs_for_hausman = sm.OLS(df['gdp_growth'], X_fs_full).fit()
fs_residuals = fs_for_hausman.resid

# Step 2: Add residuals to structural equation and test significance
X_hausman = sm.add_constant(df[['gdp_growth'] + controls])
X_hausman = X_hausman.copy()
X_hausman['fs_residuals'] = fs_residuals.values

hausman_model = sm.OLS(df['conflict_onset'], X_hausman).fit(cov_type='HC1')

hausman_t = hausman_model.tvalues['fs_residuals']
hausman_p = hausman_model.pvalues['fs_residuals']
hausman_f = hausman_t ** 2  # F = t^2 for single restriction

print("Durbin-Wu-Hausman Test for Endogeneity")
print("=" * 50)
print(f"Coefficient on first-stage residuals: {hausman_model.params['fs_residuals']:.5f}")
print(f"t-statistic: {hausman_t:.3f}")
print(f"F-statistic: {hausman_f:.3f}")
print(f"p-value: {hausman_p:.4f}")
print(f"\nNull hypothesis: GDP growth is exogenous (OLS is consistent)")
if hausman_p < 0.05:
    print(f"Result: Reject at 5% level — evidence of endogeneity; IV is preferred")
elif hausman_p < 0.10:
    print(f"Result: Reject at 10% level — some evidence of endogeneity")
else:
    print(f"Result: Fail to reject — no strong evidence of endogeneity")

## Question 7: Robustness and Policy Implications

### 7(a) Subsample Analysis by Ethnic Fractionalization

In [ ]:
median_ef = df['ethnic_fractionalization'].median()
print(f"Median ethnic fractionalization: {median_ef:.3f}")

df_high = df[df['ethnic_fractionalization'] >= median_ef].copy()
df_low = df[df['ethnic_fractionalization'] < median_ef].copy()

print(f"Above-median subsample: {len(df_high)} obs, {df_high['country_id'].nunique()} countries")
print(f"Below-median subsample: {len(df_low)} obs, {df_low['country_id'].nunique()} countries")

for label, subset in [('Above-Median Ethnic Fractionalization', df_high),
                       ('Below-Median Ethnic Fractionalization', df_low)]:
    exog_sub = sm.add_constant(subset[controls])
    try:
        iv_sub = IV2SLS(
            dependent=subset['conflict_onset'],
            exog=exog_sub,
            endog=subset[['gdp_growth']],
            instruments=subset[['rainfall_growth', 'rainfall_growth_lag']]
        ).fit(cov_type='robust')
        print(f"\n--- {label} ---")
        print(f"IV Coefficient on GDP Growth: {iv_sub.params['gdp_growth']:.5f}")
        print(f"Robust Std. Error: {iv_sub.std_errors['gdp_growth']:.5f}")
        print(f"t-statistic: {iv_sub.params['gdp_growth']/iv_sub.std_errors['gdp_growth']:.3f}")
        p_val = 2 * (1 - stats.norm.cdf(abs(iv_sub.params['gdp_growth']/iv_sub.std_errors['gdp_growth'])))
        print(f"p-value: {p_val:.4f}")
        print(f"N = {int(iv_sub.nobs)}")
    except Exception as e:
        print(f"\n--- {label} ---")
        print(f"Error: {e}")

### Summary Table: Comparing All Specifications

In [ ]:
print("\n" + "=" * 75)
print("SUMMARY TABLE: Estimated Effect of GDP Growth on Conflict Onset")
print("=" * 75)
print(f"{'Specification':<40} {'Coeff':>10} {'SE':>10} {'p-val':>10}")
print("-" * 75)
print(f"{'OLS (baseline)':<40} {ols_model.params['gdp_growth']:>10.5f} {ols_model.bse['gdp_growth']:>10.5f} {ols_model.pvalues['gdp_growth']:>10.4f}")
print(f"{'IV: 1 instrument (rainfall)':<40} {iv1_model.params['gdp_growth']:>10.5f} {iv1_model.std_errors['gdp_growth']:>10.5f} {iv1_model.pvalues['gdp_growth']:>10.4f}")
print(f"{'IV: 2 instruments (rainfall + lag)':<40} {iv2_model.params['gdp_growth']:>10.5f} {iv2_model.std_errors['gdp_growth']:>10.5f} {iv2_model.pvalues['gdp_growth']:>10.4f}")
print("-" * 75)
print(f"{'First-stage F (1 instrument)':<40} {f_stat_1:>10.2f}")
print(f"{'First-stage F (2 instruments)':<40} {f_stat_2:>10.2f}")
print(f"{'Hansen J stat (p-value)':<40} {j_stat:>10.4f} {'':>10} {j_pval:>10.4f}")
print(f"{'Hausman test (p-value)':<40} {hausman_f:>10.3f} {'':>10} {hausman_p:>10.4f}")
print("=" * 75)